# CISC 473 P08 — run genuine experiments on a Colab GPU

1. Choose **Runtime → Change runtime type → T4 GPU**, then connect.
2. Run cells in order. Authorize Google Drive when prompted. A separate Python 3.12.12 interpreter and isolated environment avoid changing Colab's preinstalled ML stack; newer notebook-kernel versions are supported.
3. Start with the smoke cell. Then run one research configuration at a time using the selectors. Each completed run is preserved on Drive.
4. Repeat for all three seeds and seven configurations, or deliberately select `all_remaining` to run sequentially until the session ends. The complete matrix can take multiple sessions.

This notebook contains no benchmark results. Smoke outputs are execution checks only. Training uses 10,000 fixed CNN/DailyMail examples, three epochs, full validation, and FP16 for every research configuration. Full test evaluation follows each run without further training. Code and Hub artifacts are pinned below. Keep the same GPU type and environment for comparisons.


In [ ]:
# 1. Mount persistent storage and select the study directory.
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
STUDY = 'midterm_fp16_v1'  # Use a new name for a different experimental protocol.
ROOT = Path('/content/drive/MyDrive/CISC473_P08') / STUDY
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
print('Persistent results:', ROOT)


In [ ]:
# 2. Install the tested code into an isolated environment (no kernel restart required).
CODE_REVISION = '6c10d0a09b9a4faa5eccf4cbb32276eb640091f0'  # Merged, tested experiment implementation.
MODEL_REVISION = 'aadd2ab0ae0c8268c7c9693540e9904811f36177'
CNN_REVISION = '96df5e686bee6baa90b8bee7c28b81fa3fa6223d'
XSUM_REVISION = '7d4d486c2f8ef850b1a11aead99b894ff3dd7da9'
REPO = Path('/content/DeepLearningP08')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/wenbin05/DeepLearningP08.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', CODE_REVISION], check=True)
# Colab's notebook kernel may use Python 3.13+. Install a separate pinned
# interpreter for this project's dependencies; leave the notebook kernel alone.
PYTHON_VERSION = '3.12.12'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.4'], check=True)
BOOTSTRAP_ENV = dict(os.environ, UV_NATIVE_TLS='true')
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', PYTHON_VERSION],
               env=BOOTSTRAP_ENV, check=True)
VENV = Path('/content/p08-py312')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION,
                    '--seed', str(VENV)], env=BOOTSTRAP_ENV, check=True)
PYTHON = str(VENV / 'bin/python')
project_version = subprocess.check_output(
    [PYTHON, '-c', 'import platform; print(platform.python_version())'], text=True).strip()
if project_version != PYTHON_VERSION:
    raise RuntimeError(f'Expected project Python {PYTHON_VERSION}, got {project_version}. Use a fresh runtime.')
print('Colab kernel:', sys.version.split()[0], '| Project Python:', project_version)
LOCK = ROOT / 'environment-lock.txt'
requirements = LOCK if LOCK.exists() else REPO / 'requirements.txt'
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
if not LOCK.exists():
    LOCK.write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze', '--all'], text=True))
subprocess.run([PYTHON, '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"); assert torch.cuda.is_available(), "Select a GPU runtime"'], check=True)
ENV = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')

# Refuse to combine studies with different declared protocols.
gpu_name = subprocess.check_output([PYTHON, '-c', 'import torch; print(torch.cuda.get_device_name())'], text=True).strip()
protocol = dict(code=CODE_REVISION, model=MODEL_REVISION, cnn=CNN_REVISION,
                xsum=XSUM_REVISION, precision='fp16', train_samples=10000,
                epochs=3, warmup_steps=500, data_seed=42,
                python=subprocess.check_output([PYTHON, '-c', 'import sys; print(sys.version)'], text=True).strip(),
                gpu=gpu_name)
protocol_path = ROOT / 'protocol.json'
if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
    raise RuntimeError('Study protocol changed: choose a new STUDY directory.')
protocol_path.write_text(json.dumps(protocol, indent=2) + '\n')

# Stream subprocess output to the cell and a persistent log; propagate every failure.
def run(arguments, log_name):
    logs = ROOT / 'logs'
    logs.mkdir(exist_ok=True)
    with (logs / log_name).open('a') as log:
        process = subprocess.Popen([PYTHON, '-u', *map(str, arguments)], cwd=REPO,
            env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            if process.wait():
                raise RuntimeError(f'Command failed. See {logs / log_name}')
        except BaseException:
            process.terminate()
            process.wait()
            raise


In [ ]:
# 3. Real-model smoke check. Outputs here are NOT research results.
# Change SMOKE_ID only if you deliberately want a fresh execution check.
SMOKE_ID = 'check_1'
smoke_root = ROOT / 'smoke' / SMOKE_ID
smoke_run = smoke_root / 'cross_attn_r8_q_proj,v_proj_seed42'
if (smoke_run / 'validation_results.json').exists():
    print('Smoke already completed:', smoke_run)
elif smoke_run.exists():
    raise RuntimeError('Smoke run was interrupted. Set a new SMOKE_ID to retry; existing files are preserved.')
else:
    run(['-m', 'src.train', '--placement', 'cross_attn', '--seed', '42',
         '--max_train_samples', '8', '--max_eval_samples', '2', '--max_steps', '1',
         '--warmup_steps', '0', '--fp16', '--model_revision', MODEL_REVISION,
         '--dataset_revision', CNN_REVISION, '--output_dir', smoke_root], 'smoke.log')
print(json.loads((smoke_run / 'validation_results.json').read_text())['metrics'])


## Research runs

The next cell runs real experiments and may take hours. `single` is the recommended starting mode. For each placement, run seeds **42, 123, 999**. Baselines are `zero_shot`, `full_ft`, `prefix`, and `all_attn`; only the three isolated placements share a parameter budget.

Completed runs are skipped after checking their saved protocol. Completed test evaluations are also skipped. An interrupted **training** run is preserved and explicitly stops this launcher: automatic mid-epoch training resume is not implemented. To retry, move that incomplete run folder to an archive directory **outside** `research` on Drive, then rerun the same selection. If only **test evaluation** was interrupted, move that incomplete evaluation subfolder outside `research` and rerun; training will be skipped.

Do not change sample limits to obtain report numbers faster. Keep `smoke` separate from `research`. Do not launch two sessions against the same run directory.


In [ ]:
# 4. Select and execute research runs, followed by both full test evaluations.
PLACEMENT = 'cross_attn' #@param ['cross_attn', 'encoder_self_attn', 'decoder_self_attn', 'all_attn', 'full_ft', 'prefix', 'zero_shot']
SEED = 42 #@param [42, 123, 999] {type:'raw'}
RUN_MODE = 'single' #@param ['single', 'all_remaining']
research_root = ROOT / 'research'

modes = ['zero_shot', 'full_ft', 'prefix', 'all_attn',
         'encoder_self_attn', 'decoder_self_attn', 'cross_attn']
selected = [(PLACEMENT, SEED)] if RUN_MODE == 'single' else [
    (mode, seed) for seed in [42, 123, 999] for mode in modes]
for mode, seed in selected:
    directory = research_root / f'{mode}_r8_q_proj,v_proj_seed{seed}'
    completed = directory / 'validation_results.json'
    if completed.exists():
        previous = json.loads(completed.read_text())
        c = previous['manifest']['config']
        expected = dict(placement=mode, lora_rank=8, projections='q_proj,v_proj',
                        model_revision=MODEL_REVISION, dataset_revision=CNN_REVISION,
                        fp16=True, data_seed=42, num_train_epochs=3, warmup_steps=500)
        if previous.get('status') != 'completed' or previous['manifest']['seed'] != seed or any(c[k] != v for k, v in expected.items()):
            raise RuntimeError(f'Saved run has a different protocol: {directory}')
        print('Training completed previously:', directory)
    elif directory.exists():
        raise RuntimeError(f'Incomplete training run preserved at {directory}. See retry instructions above.')
    else:
        run(['-m', 'src.train', '--placement', mode, '--seed', seed, '--fp16',
             '--model_revision', MODEL_REVISION, '--dataset_revision', CNN_REVISION,
             '--output_dir', research_root], f'{mode}_seed{seed}_train.log')
    for dataset, revision in [('cnn_dailymail', CNN_REVISION), ('xsum', XSUM_REVISION)]:
        files = list((directory / 'evaluations').glob(f'{dataset}_*/test_results.json'))
        if files:
            saved = json.loads(files[0].read_text())
            meta = saved['manifest']['evaluation']
            if len(files) != 1 or saved.get('status') != 'completed' or meta['revision'] != revision or meta['split'] != 'test' or meta['samples'] != meta['official_size']:
                raise RuntimeError(f'Unexpected saved evaluation protocol: {directory}')
            print('Full test evaluation completed previously:', files[0])
        else:
            run(['-m', 'src.train', '--eval_only', directory, '--eval_dataset', dataset,
                 '--eval_revision', revision], f'{mode}_seed{seed}_{dataset}.log')
print('Selected runs completed. All artifacts remain on Google Drive:', research_root)


In [ ]:
# 5. Show completion status. This does not assume any experiment succeeded.
for seed in [42, 123, 999]:
    for mode in modes:
        folder = research_root / f'{mode}_r8_q_proj,v_proj_seed{seed}'
        validation = (folder / 'validation_results.json').exists()
        tests = {name: bool(list((folder / 'evaluations').glob(f'{name}_*/test_results.json')))
                 for name in ['cnn_dailymail', 'xsum']}
        print(seed, mode, 'validation:', validation, 'tests:', tests)


In [ ]:
# 6. Aggregate actual saved outputs; missing seeds cause a clear error.
# Run after each configuration has all three completed seeds and test evaluations.
run(['-m', 'src.aggregate_results', '--results_dir', research_root,
     '--phase', 'test', '--output', research_root / 'summary.json'], 'aggregate.log')
for dataset in ['cnn_dailymail', 'xsum']:
    run(['-m', 'src.plot_results', '--summary', research_root / 'summary.json',
         '--dataset', dataset, '--output', research_root / f'{dataset}_pareto.png'], 'plot.log')
print('Statistics and plots:', research_root)


## After execution

Keep each complete run directory, not just its plot. Send the genuine `validation_results.json`, `test_results.json`, and `summary.json` files back to Codex for analysis and report updates. Do not present execution-check scores as benchmark findings. Additional rank/projection ablations and the human factuality study remain separate work. Equal updates are not proof of equal wall-clock or FLOP budgets.
